In [1]:
!pip install yfinance torch


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [3]:
df = yf.download("AAPL", period="6mo", interval="1d")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df = df.reset_index()
df = df.sort_values("Date")

[*********************100%***********************]  1 of 1 completed


In [4]:
df["sma_10"] = df["Close"].rolling(10).mean()
df["sma_20"] = df["Close"].rolling(20).mean()

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

rolling_mean = df["Close"].rolling(20).mean()
rolling_std  = df["Close"].rolling(20).std()
df["bb_upper"] = rolling_mean + (2 * rolling_std)
df["bb_lower"] = rolling_mean - (2 * rolling_std)
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]

df["return"]     = df["Close"].pct_change()
df["return_3d"]  = df["Close"].pct_change(3)
df["return_5d"]  = df["Close"].pct_change(5)

df["volatility_5d"]  = df["return"].rolling(5).std()
df["volatility_10d"] = df["return"].rolling(10).std()

df["dist_sma_10"] = (df["Close"] - df["sma_10"]) / df["sma_10"]
df["dist_sma_20"] = (df["Close"] - df["sma_20"]) / df["sma_20"]

df["target"] = (df["return"].shift(-1) > 0.002).astype(int)
df = df.dropna()

In [5]:
feature_cols = [
    "rsi_14", "macd", "macd_signal", "bb_width",
    "return_3d", "return_5d", "volatility_5d", "volatility_10d",
    "dist_sma_10", "dist_sma_20", "Volume"
]

X = df[feature_cols].values
y = df["target"].values

split_index = int(len(df) * 0.8)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

LOOKBACK = 10

def create_sequences(X, y, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y, LOOKBACK)

train_size = split_index - LOOKBACK

X_train = torch.tensor(X_seq[:train_size], dtype=torch.float32)
X_test  = torch.tensor(X_seq[train_size:], dtype=torch.float32)
y_train = torch.tensor(y_seq[:train_size], dtype=torch.float32)
y_test  = torch.tensor(y_seq[train_size:], dtype=torch.float32)

train_dl = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=False)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: torch.Size([74, 10, 11]), Test: torch.Size([22, 10, 11])


In [6]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(hidden_size, 32, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.fc    = nn.Linear(32, 16)
        self.relu  = nn.ReLU()
        self.out   = nn.Sequential(nn.Linear(16, 1), nn.Sigmoid())

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.drop1(x)
        x, _ = self.lstm2(x)
        x = self.drop2(x[:, -1, :])
        x = self.relu(self.fc(x))
        return self.out(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = LSTMClassifier(input_size=len(feature_cols)).to(device)
print(model)

LSTMClassifier(
  (lstm1): LSTM(11, 64, batch_first=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (lstm2): LSTM(64, 32, batch_first=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=32, out_features=16, bias=True)
  (relu): ReLU()
  (out): Sequential(
    (0): Linear(in_features=16, out_features=1, bias=True)
    (1): Sigmoid()
  )
)


c:\Users\deera\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10\cuda\CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

best_val_loss = float("inf")
patience, wait = 10, 0
best_weights   = None

X_test_dev = X_test.to(device)
y_test_dev = y_test.to(device)

for epoch in range(1, 101):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_test_dev), y_test_dev).item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | val_loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_weights)

Epoch  10 | val_loss: 0.6656
Early stopping at epoch 15


<All keys matched successfully>

In [8]:
model.eval()
with torch.no_grad():
    prob_t = model(X_test_dev)

sig_prob = prob_t.cpu().numpy()
sig_pred = (sig_prob > 0.5).astype(int)

print("LSTM Accuracy:", accuracy_score(y_test.numpy(), sig_pred))
print(classification_report(y_test.numpy(), sig_pred))
print("ROC-AUC:", roc_auc_score(y_test.numpy(), sig_prob))

LSTM Accuracy: 0.6363636363636364
              precision    recall  f1-score   support

         0.0       0.64      1.00      0.78        14
         1.0       0.00      0.00      0.00         8

    accuracy                           0.64        22
   macro avg       0.32      0.50      0.39        22
weighted avg       0.40      0.64      0.49        22

ROC-AUC: 0.7410714285714286


c:\Users\deera\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\deera\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\deera\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [9]:
prob = float(sig_prob[-1])

if prob > 0.6:
    signal = "BUY"
elif prob < 0.4:
    signal = "SELL"
else:
    signal = "HOLD"

print(f"Signal:     {signal}")
print(f"Confidence: {round(prob, 2)}")

Signal:     HOLD
Confidence: 0.44
